Once your code outgrows a single script, package it. A proper package can be `pip install`-ed, imported from anywhere, versioned, and shared with collaborators — no more `sys.path` hacks or copy-pasted files.

::: {.callout-tip}
## Why learn this?
Every group has that one folder of copy-pasted `utils.py` files that have quietly drifted apart. Packaging turns your scattered scripts into a single, versioned toolkit that you (and your labmates) `pip install` once and import everywhere.

- **Imagine you need** the same `tke()` in five different projects — put it in a package instead of copy-pasting it five times.
- **Imagine you need** to hand your tools to a collaborator — `pip install` from your repo and they have everything.
:::

## The modern project layout
The recommended *src layout* keeps importable code under `src/` and metadata in a single `pyproject.toml`:

```
myturb/
├── pyproject.toml        # name, version, dependencies, build backend
├── README.md
└── src/
    └── myturb/
        ├── __init__.py    # marks the package; sets __version__, public API
        └── stats.py       # a module
```

A minimal `pyproject.toml`:

```toml
[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "myturb"
version = "0.1.0"
description = "Small turbulence post-processing helpers"
requires-python = ">=3.10"
dependencies = ["numpy"]
```

## Build a real package here
The cell below **writes that layout to a temporary folder**, so you can see the files a package is made of. (In a normal project these live in your repo.)

In [1]:
from pathlib import Path
import tempfile, textwrap, sys

root = Path(tempfile.mkdtemp()) / 'myturb'
pkg = root / 'src' / 'myturb'
pkg.mkdir(parents=True)

(root / 'pyproject.toml').write_text(textwrap.dedent('''
    [build-system]
    requires = ["hatchling"]
    build-backend = "hatchling.build"

    [project]
    name = "myturb"
    version = "0.1.0"
    requires-python = ">=3.10"
'''))

(pkg / '__init__.py').write_text(textwrap.dedent('''
    """myturb - tiny turbulence helpers."""
    __version__ = '0.1.0'
    from .stats import tke          # expose a clean public API
'''))

(pkg / 'stats.py').write_text(textwrap.dedent('''
    import numpy as np
    def tke(u, v, w):
        """Turbulent kinetic energy from fluctuation arrays."""
        u, v, w = map(np.asarray, (u, v, w))
        return 0.5 * (u**2 + v**2 + w**2).mean()
'''))

for p in sorted(root.rglob('*')):
    print(p.relative_to(root.parent))

myturb/pyproject.toml
myturb/src
myturb/src/myturb
myturb/src/myturb/__init__.py
myturb/src/myturb/stats.py


## Import and use it
Normally you would run `pip install -e .` from the project root (an *editable* install) so the package is importable everywhere. Here we mimic that by adding `src/` to the import path.

In [2]:
import sys, importlib
sys.path.insert(0, str(root / 'src'))
import myturb
importlib.reload(myturb)

print('version:', myturb.__version__)
print('tke     :', round(myturb.tke([1, -1], [2, -2], [0, 0]), 3))

version: 0.1.0
tke     : 2.5


**Imagine you need** to keep editing your library while a dozen scripts already import it. This is super easy in Python using an *editable install*, `pip install -e .`.

## Installing, building, and distributing
The commands you would actually run from the project root (shown, not executed here):


In [ ]:
#| eval: false
# editable install for development (changes picked up immediately)
pip install -e .

# build distributables (wheel + sdist) into dist/
python -m build

# upload to (Test)PyPI with twine
python -m twine upload --repository testpypi dist/*

Good habits for a shareable package: pin a `requires-python`, list runtime `dependencies`, add a `README.md` and a `LICENSE`, keep a single source of truth for the version, and add tests (see [Software engineering practices](06-engineering.ipynb)).

## Self-tests

**1.** What is the purpose of `__init__.py` in a package directory?

::: {.callout-tip collapse="true"}
## Solution
It marks the directory as a Python **package** and runs when the package is imported — a natural place to set `__version__` and to re-export the public API (e.g. `from .stats import tke`) so users can write `from myturb import tke`.
:::

**2.** Add a `moments.py` module to the temp package with a function `rms(x)` returning the root-mean-square, expose it in `__init__`, and call it.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| output: false
import textwrap, importlib
(pkg / 'moments.py').write_text(textwrap.dedent('''
    import numpy as np
    def rms(x):
        x = np.asarray(x)
        return float(np.sqrt((x**2).mean()))
'''))
# re-expose and reload
init = (pkg / '__init__.py')
init.write_text(init.read_text() + 'from .moments import rms\n')
import myturb, importlib
importlib.reload(myturb)
print(myturb.rms([3, 4]))   # -> 3.5355...

## Turbulence in practice: grow the package

A package is meant to accumulate your toolkit. Add a `reynolds_stress` helper to `stats.py`, re-expose it in the public API, and use it.

In [4]:
import importlib, textwrap
import myturb, myturb.stats

stats = pkg / 'stats.py'
stats.write_text(stats.read_text() + textwrap.dedent('''
    def reynolds_stress(u, v):
        # Reynolds shear stress <u'v'> from two fluctuation arrays
        u, v = map(np.asarray, (u, v))
        return float((u * v).mean())
'''))
init = pkg / '__init__.py'
init.write_text(init.read_text() + 'from .stats import reynolds_stress\n')

importlib.reload(myturb.stats)   # refresh the submodule first...
importlib.reload(myturb)         # ...then re-run the package __init__
print("<u'v'> =", myturb.reynolds_stress([1, -1, 2], [2, -2, 1]))

<u'v'> = 2.0


**Self-test.** Add a `turbulence_intensity(u_rms, u_mean)` function to `stats.py`, expose it, and call it.

In [ ]:
#| code-fold: true
#| code-summary: "Show solution"
#| output: false
stats.write_text(stats.read_text()
    + '\ndef turbulence_intensity(u_rms, u_mean):\n'
    + '    return u_rms / u_mean\n')
init.write_text(init.read_text()
    + 'from .stats import turbulence_intensity\n')
import importlib
importlib.reload(myturb.stats)
importlib.reload(myturb)
print(myturb.turbulence_intensity(1.1, 10.0))